<a href="https://colab.research.google.com/github/seoyeon-ss/osp-project/blob/main/training_py.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip uninstall -y ultralytics
!git clone https://github.com/sunsmarterjie/yolov12.git
%cd /content/yolov12
!pip install -r requirements.txt
!pip install -e .

Cloning into 'yolov12'...
remote: Enumerating objects: 1173, done.
remote: Counting objects: 100% (1/1), done.
remote: Total 1173 (delta 0), reused 0 (delta 0), pack-reused 1172 (from 2)
Receiving objects: 100% (1173/1173), 1.95 MiB | 6.51 MiB/s, done.
Resolving deltas: 100% (531/531), done.
/content/yolov12
ERROR: flash_attn-2.7.3+cu11torch2.2cxx11abiFALSE-cp311-cp311-linux_x86_64.whl is not a supported wheel on this platform.
Obtaining file:///content/yolov12
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for ultralytics (pyproject.toml) ... done
  Created wheel for ultralytics: filename=ultralytics-8.3.63-0.editable-py3-none-any.whl size=20189 sha256=2e687b9991cb655f8e9986034080bbc06732881dc8aa163bcfb4aa7529a1fdb8
  Stored in directory: /tmp/pip-ephem-wheel-cache-teftzwdv/wheels/1c/fb/0a/30d0595ef49b9e095

In [ ]:
!wget -O /content/yolov12n-cls.pt \
https://github.com/sunsmarterjie/yolov12/releases/download/cls/yolov12n-cls.pt

--2026-06-20 09:46:36--  https://github.com/sunsmarterjie/yolov12/releases/download/cls/yolov12n-cls.pt
Resolving github.com (github.com)... 140.82.113.3
Connecting to github.com (github.com)|140.82.113.3|:443... connected.
HTTP request sent, awaiting response... 302 Found
Location: https://release-assets.githubusercontent.com/github-production-release-asset/928546208/60394958-303e-4d58-a22f-0b2a1760a894?sp=r&sv=2018-11-09&sr=b&spr=https&se=2026-06-20T10%3A26%3A27Z&rscd=attachment%3B+filename%3Dyolov12n-cls.pt&rsct=application%2Foctet-stream&skoid=96c2d410-5711-43a1-aedd-ab1947aa7ab0&sktid=398a6654-997b-47e9-b12b-9515b896b4de&skt=2026-06-20T09%3A25%3A44Z&ske=2026-06-20T10%3A26%3A27Z&sks=b&skv=2018-11-09&sig=EUWW3VkY8PY5rGNJ5nwmyy0y1lufWvwlv5%2BiIaWr0NQ%3D&jwt=eyJ0eXAiOiJKV1QiLCJhbGciOiJIUzI1NiJ9.eyJpc3MiOiJnaXRodWIuY29tIiwiYXVkIjoicmVsZWFzZS1hc3NldHMuZ2l0aHVidXNlcmNvbnRlbnQuY29tIiwia2V5Ijoia2V5MSIsImV4cCI6MTc4MTk0OTA5NiwibmJmIjoxNzgxOTQ4Nzk2LCJwYXRoIjoicmVsZWFzZWFzc2V0cHJvZHVjdGlvbi5ib

In [ ]:
!ls -lh /content/yolov12n-cls.pt

-rw-r--r-- 1 root root 5.9M Jul  1  2025 /content/yolov12n-cls.pt


In [ ]:
from google.colab import files

uploaded = files.upload()

Saving Dataset.zip to Dataset.zip


In [ ]:
%cd /content
!unzip -q Dataset.zip

/content
unzip:  cannot find or open Dataset.zip, Dataset.zip.zip or Dataset.zip.ZIP.


In [ ]:
!ls -lh /content

total 5.9M
drwxr-xr-x  1 root root 4.0K Jun  4 13:39 sample_data
drwxr-xr-x 10 root root 4.0K Jun 20 09:49 yolov12
-rw-r--r--  1 root root 5.9M Jul  1  2025 yolov12n-cls.pt


In [ ]:
from google.colab import files

uploaded = files.upload()
print(uploaded.keys())

Saving Dataset.zip to Dataset.zip
dict_keys(['Dataset.zip'])


In [ ]:
uploaded_name = next(iter(uploaded))
print("업로드된 파일:", uploaded_name)

!unzip -q "/content/{uploaded_name}" -d /content

업로드된 파일: Dataset.zip


In [ ]:
!ls /content/Dataset

test  train  valid


In [ ]:
from pathlib import Path

valid_path = Path("/content/Dataset/valid")
val_path = Path("/content/Dataset/val")

if valid_path.exists() and not val_path.exists():
    valid_path.rename(val_path)

print(list(Path("/content/Dataset").iterdir()))

[PosixPath('/content/Dataset/test'), PosixPath('/content/Dataset/.DS_Store'), PosixPath('/content/Dataset/val'), PosixPath('/content/Dataset/train')]


In [ ]:
!ls /content/Dataset

test  train  val


In [ ]:
import torch

print("CUDA 사용 가능:", torch.cuda.is_available())
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "없음")

CUDA 사용 가능: True
GPU: Tesla T4


In [ ]:
from pathlib import Path

dataset_path = Path("/content/Dataset")

for split in ["train", "val", "test"]:
    classes = sorted(
        folder.name
        for folder in (dataset_path / split).iterdir()
        if folder.is_dir()
    )
    print(split, len(classes), classes)

train 15 ['banana', 'chocolate', 'fried-egg', 'gimbap', 'grilled salmon', 'jajangmyeon', 'kimchi', 'ramen', 'red apple', 'salad', 'sandwich', 'sweetpotato', 'tteokbokki', 'waffle', 'white rice']
val 15 ['banana', 'chocolate', 'fried-egg', 'gimbap', 'grilled salmon', 'jajangmyeon', 'kimchi', 'ramen', 'red apple', 'salad', 'sandwich', 'sweetpotato', 'tteokbokki', 'waffle', 'white rice']
test 15 ['banana', 'chocolate', 'fried-egg', 'gimbap', 'grilled salmon', 'jajangmyeon', 'kimchi', 'ramen', 'red apple', 'salad', 'sandwich', 'sweetpotato', 'tteokbokki', 'waffle', 'white rice']


In [ ]:
!ls -lh /content/yolov12n-cls.pt

-rw-r--r-- 1 root root 5.9M Jul  1  2025 /content/yolov12n-cls.pt


In [ ]:
%cd /content/yolov12

from ultralytics import YOLO

model = YOLO("/content/yolov12n-cls.pt")

results = model.train(
    data="/content/Dataset",
    epochs=50,
    imgsz=224,
    batch=16,
    device=0,
    workers=2,
    project="/content/runs/classify",
    name="food_yolov12n"
)

/content/yolov12
Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/yolov12/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
FlashAttention is not available on this device. Using scaled_dot_product_attention instead.
New https://pypi.org/project/ultralytics/8.4.72 available 😃 Update with 'pip install -U ultralytics'
Ultralytics 8.3.63 🚀 Python-3.12.13 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: task=classify, mode=train, model=/content/yolov12n-cls.pt, data=/content/Dataset, epochs=50, time=None, patience=100, batch=16, imgsz=224, save=True, save_period=-1, cache=False, device=0, workers=2, project=/content/runs/classify, name=food_yolov12n, exist_ok=False, pretrained=True, optimizer=auto, verbose=True, seed=0, deterministic=True, single_cls=False, rect=False, cos_lr=Fals

100%|██████████| 5.26M/5.26M [00:00<00:00, 77.0MB/s]


AMP: checks passed ✅


train: Scanning /content/Dataset/train... 1940 images, 0 corrupt: 100%|██████████| 1940/1940 [00:00<00:00, 5281.44it/s]

train: New cache created: /content/Dataset/train.cache


...

      50/50     0.375G     0.0155          4        224: 100%|██████████| 122/122 [00:17<00:00,  6.91it/s]
               classes   top1_acc   top5_acc: 100%|██████████| 11/11 [00:00<00:00, 16.29it/s]

                   all      0.985          1



50 epochs completed in 0.280 hours.
Optimizer stripped from /content/runs/classify/food_yolov12n/weights/last.pt, 3.6MB
Optimizer stripped from /content/runs/classify/food_yolov12n/weights/best.pt, 3.6MB

Validating /content/runs/classify/food_yolov12n/weights/best.pt...
Ultralytics 8.3.63 🚀 Python-3.12.13 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
YOLOv12n-cls summary (fused): 217 layers, 1,680,471 parameters, 0 gradients, 3.0 GFLOPs
train: /content/Dataset/train... found 1940 images in 15 classes ✅ 
val: /content/Dataset/val... found 324 images in 15 classes ✅ 
test: /content/Dataset/test... found 177 images in 15 classes ✅ 


               classes   top1_acc   top5_acc: 100%|██████████| 11/11 [00:01<00:00,  9.88it/s]


                   all      0.991          1
Speed: 0.4ms preprocess, 2.3ms inference, 0.0ms loss, 0.0ms postprocess per image
Results saved to /content/runs/classify/food_yolov12n
